In [1]:
import pandas as pd


In [ ]:
movies = pd.read_csv('movies_cleanedFull.csv')

In [4]:
#make two genres for zero shot classification 
entertainment_genres = {
    "Action", "Adventure", "Comedy", "Animation", "Fantasy",
    "Family", "Romance", "Musical", "Music", "Sport",
    "Game-Show", "Reality-TV", "Talk-Show", "Short",
    "Western", "Sci-Fi"
}

serious_genres = {
    "Drama", "Crime", "Thriller", "Horror", "Mystery",
    "Biography", "History", "War", "Documentary",
    "Film-Noir", "News"
}  

#create a regex strings for better search
entertainment_pattern = "|".join(entertainment_genres)
serious_pattern = "|".join(serious_genres)

movies["simplified_categories"] = "other"
movies.loc[movies["genre"].str.contains(entertainment_pattern, case=False, na=False), "simplified_categories"] = "entertainment"
movies.loc[movies["genre"].str.contains(serious_pattern, case=False, na=False),  "simplified_categories"] = "serious"
    

In [9]:
movies.to_csv("text-classMovies.csv", index=False)

In [4]:
%pip install transformers

   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ----------------- ---------------------- 5.2/12.0 MB 31.7 MB/s eta 0:00:01
   ---------------------------------------- 12.0/12.0 MB 30.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/566.1 kB ? eta -:--:--
   ---------------------------------------- 566.1/566.1 kB ? eta 0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 26.4 MB/s eta 0:00:00

   -------- ------------------------------- 2/9 [regex]
   ------------- -------------------------- 3/9 [pyyaml]
   ----------------- ---------------------- 4/9 [fsspec]
   -------------------------- ------------- 6/9 [huggingface-hub]
   -------------------------- ------------- 6/9 [huggingface-hub]
   -------------------------- ------------- 6/9 [huggingface-hub]
   -------------------------- ------------- 6/9 [huggingface-hub]
   ----------------------------------- ---- 8/9 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: C:\Users\ruba1\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [8]:
%pip install torch

  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
   ---------------------------------------- 0.0/113.7 MB ? eta -:--:--
    --------------------------------------- 2.1/113.7 MB 10.7 MB/s eta 0:00:11
   -- ------------------------------------- 6.0/113.7 MB 14.8 MB/s eta 0:00:08
   --- ------------------------------------ 9.7/113.7 MB 16.3 MB/s eta 0:00:07
   ----- ---------------------------------- 14.7/113.7 MB 18.1 MB/s eta 0:00:06
   ------- -------------------------------- 21.0/113.7 MB 20.4 MB/s eta 0:00:05
   --------- ------------------------------ 26.7/113.7 MB 21.7 MB/s eta 0:00:05
   ----------- ---------------------------- 33.6/113.7 MB 23.2 MB/s eta 0:00:04
   -------------- ------------------------- 40.6/113.7 MB 24.6 MB/s eta 0:00:03
   ----------------- ---------------------- 49.0/113.7 MB 26.4 MB/s eta 0:00:03
   ------------------- -------------------- 55.3/113.7 MB 26.7 MB/s eta 0:00:03
   --------------------- ------------------ 61.9/113.7 MB 27.0 MB/s

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: C:\Users\ruba1\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [ ]:
import torch

In [100]:

from transformers import pipeline

genre_categories = [
    "Entertainment: light, fun, action, comedy, fantasy, adventure, action, light-hearted",
    "Serious: drama, biography, history, crime, war, social issues"
]


pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device="cpu")

Device set to use cpu


In [104]:
sequence = movies.loc[movies["simplified_categories"] == "entertainment", "description"].reset_index(drop=True)[0]

In [147]:
#check which is most likely label, serious or entertainment
import numpy as np
max_index = np.argmax(pipe(sequence, genre_categories)["scores"])
max_label = pipe(sequence, genre_categories)["labels"][max_index]
max_label 

'Entertainment: light, fun, action, comedy, fantasy, adventure'

In [148]:
#run zero shot classification and extract max label

def generate_predictions(sequence, genre_categories):
    predictions = pipe(sequence, genre_categories)
    max_index = np.argmax(predictions["scores"])
    max_label = predictions["labels"][max_index]
    return max_label
    

In [130]:
from tqdm import tqdm

actual_cats = []
predicted_cats = []

for i in tqdm(range(0,300)):
    sequence = movies.loc[movies["simplified_categories"] == "entertainment", "description"].reset_index(drop=True)[i]
    predicted_cats += [generate_predictions(sequence, genre_categories)]
    actual_cats += ["entertainment"]
#     

100%|██████████| 300/300 [04:20<00:00,  1.15it/s]


In [131]:
for i in tqdm(range(0,300)):
    sequence = movies.loc[movies["simplified_categories"] == "serious", "description"].reset_index(drop=True)[i]
    predicted_cats += [generate_predictions(sequence, genre_categories)]
    actual_cats += ["serious"]

100%|██████████| 300/300 [04:07<00:00,  1.21it/s]


In [132]:
predictions_df = pd.DataFrame({"actual_cats": actual_cats, "predicted_cats": predicted_cats})

In [134]:
predictions_df["correct_predictions"] = (
    np.where(predictions_df["predicted_cats"].str.lower() == predictions_df["predicted_cats"].str.lower(), 1, 0)
)

In [135]:
predictions_df["correct_predictions"] = predictions_df.apply(
    lambda row: int(
        row["predicted_cats"].lower().startswith(row["actual_cats"].lower())
    ),
    axis=1
)

In [154]:
movie_ids = []
predicted_cats = []

missing_cats = movies.loc[movies["genre"].isna(), ["movie_id", "description"]].reset_index(drop=True)

In [156]:
for i in tqdm(range(0, len(missing_cats))):
    sequence = missing_cats['description'][i]
    predicted_cats += [generate_predictions(sequence, genre_categories)]
    movie_ids += [missing_cats['movie_id'][i]]

100%|██████████| 38/38 [00:30<00:00,  1.23it/s]


In [176]:
missing_predicted_df = pd.DataFrame({"movie_id": movie_ids, "predicted_cats": predicted_cats})

In [178]:
maskS = missing_predicted_df["predicted_cats"].str.contains("serious", case=False, na=False)
maskE = missing_predicted_df["predicted_cats"].str.contains("entertainment", case=False, na=False)

missing_predicted_df.loc[maskS, "predicted_cats"] = "serious"
missing_predicted_df.loc[maskE, "predicted_cats"] = "entertainment"



In [181]:
movies = pd.merge(movies, missing_predicted_df, on="movie_id", how="left")
movies["simplified_categories"] = np.where(movies["simplified_categories"].isna(), movies["predicted_cats"], movies["simplified_categories"])
movies = movies.drop(columns=["predicted_cats"])

In [185]:
movies

,movie_id,title,year,certificate,duration,genre,rating,description,stars,votes,words_in_description,tagged_description,simplified_categories
0,1,Cobra Kai,(2018– ),TV-14,30 min,"Action, Comedy, Drama",8.5,Decades after their 1984 All Valley Karate Tou...,"['Ralph Macchio, ', 'William Zabka, ', 'Courtn...","177,031",21,1 Decades after their 1984 All Valley Karate T...,serious
1,2,The Crown,(2016– ),TV-MA,58 min,"Biography, Drama, History",8.7,Follows the political rivalries and romance of...,"['Claire Foy, ', 'Olivia Colman, ', 'Imelda St...","199,885",23,2 Follows the political rivalries and romance ...,serious
2,3,Better Call Saul,(2015–2022),TV-MA,46 min,"Crime, Drama",8.9,The trials and tribulations of criminal lawyer...,"['Bob Odenkirk, ', 'Rhea Seehorn, ', 'Jonathan...","501,384",19,3 The trials and tribulations of criminal lawy...,serious
3,4,Devil in Ohio,(2022),TV-MA,356 min,"Drama, Horror, Mystery",5.9,When a psychiatrist shelters a mysterious cult...,"['Emily Deschanel, ', 'Sam Jaeger, ', 'Gerardo...","9,773",25,4 When a psychiatrist shelters a mysterious cu...,serious
4,5,Cyberpunk: Edgerunners,(2022– ),TV-MA,24 min,"Animation, Action, Adventure",8.6,A Street Kid trying to survive in a technology...,"['Zach Aguilar, ', 'Kenichiro Ohashi, ', 'Emi ...","15,413",37,5 A Street Kid trying to survive in a technolo...,entertainment
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7551,7907,Star Trek: Discovery,(2017– ),TV-MA,49 min,"Action, Adventure, Drama",7.2,"Finally reunited, Burnham and the U.S.S. Disco...","['Jonathan Frakes', '| ', ' Stars:', 'Soneq...","3,373",22,"7907 Finally reunited, Burnham and the U.S.S. ...",serious
7552,7909,The Weekly with Wendy Mesley,(2018– ),NaN,136,"News, Reality-TV",7.6,How did Ontario's popular new Premier get so u...,"['Wendy Mesley, ', 'Billy Porter, ', 'Jacob To...",NaN,23,7909 How did Ontario's popular new Premier get...,serious
7553,7910,The Drew Barrymore Show,(2020– ),TV-PG,44 min,Talk-Show,6.0,In the inaugural episode of The Drew Barrymore...,"['Adam Heydt', '| ', ' Stars:', 'Drew Barry...",19,36,7910 In the inaugural episode of The Drew Barr...,entertainment
7554,7911,Hollywood Insider,(2018– ),NaN,136,Talk-Show,7.6,Behind the scenes of The Irishman.,"['Bobby Cannavale, ', 'Robert De Niro, ', 'Al ...",NaN,6,7911 Behind the scenes of The Irishman.,entertainment


In [186]:
movies.to_csv("movies2.csv", index=False)